## HW1 — Surgical analysis of attention heads (GPT-2)

This notebook contains **all code used** for:
- **Part A (Experiment)**: mapping → ablation (hooks) → evaluation, producing `ID_results.csv`
- **Part B (Analysis)**: a small analysis + the **required top‑5 before/after visualization**

**Inputs** (same folder):
- `ex1_data.csv`

**Outputs**:
- `ID_results.csv` (rename to include your ID(s) before submitting)


### Setup

If you haven't installed dependencies yet (Windows PowerShell):

```bash
python -m venv .venv
.\.venv\Scripts\Activate.ps1
python -m pip install --upgrade pip
pip install torch transformers pandas numpy matplotlib
```


In [ ]:
import matplotlib
matplotlib.use("TkAgg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

MODEL_NAME = "gpt2"          # GPT-2 small
DATA_PATH = "ex1_data.csv"
OUTPUT_CSV = "ID_results.csv"  # rename to include your ID(s)


In [ ]:
def get_subject_token_index(prompt, subject, tokenizer):
    """Index of the last token of `subject` inside the tokenized `prompt`."""
    inputs = tokenizer(prompt, return_tensors="pt")
    prompt_ids = inputs["input_ids"][0].tolist()
    prompt_tokens = [tokenizer.decode([tid]) for tid in prompt_ids]

    subject_ids = tokenizer.encode(subject, add_prefix_space=True)

    n = len(subject_ids)
    for i in range(len(prompt_ids) - n, -1, -1):
        if prompt_ids[i : i + n] == subject_ids:
            return i + n - 1

    # Fallback heuristic
    subject_plain = subject.strip().lower()
    for i in range(len(prompt_tokens) - 1, -1, -1):
        if subject_plain in prompt_tokens[i].strip().lower():
            return i

    return len(prompt_ids) - 2


def get_target_token_ids(target_token, tokenizer):
    """Token ids for the target as the next continuation (space-prefixed)."""
    target_token_str = " " + str(target_token).strip()
    return tokenizer.encode(target_token_str, add_special_tokens=False)


def prob_of_target_sequence(prompt, target_token_ids, model, tokenizer):
    """Probability of generating the (possibly multi-token) target after `prompt`."""
    if len(target_token_ids) == 0:
        return 0.0

    inputs = tokenizer(prompt, return_tensors="pt")
    input_ids = inputs["input_ids"]

    p = 1.0
    with torch.no_grad():
        for next_tid in target_token_ids:
            out = model(input_ids=input_ids)
            next_probs = torch.softmax(out.logits[0, -1, :], dim=-1)
            p *= next_probs[int(next_tid)].item()
            input_ids = torch.cat([input_ids, torch.tensor([[int(next_tid)]])], dim=1)

    return p


class HeadAblator:
    """Zeros selected head subspaces via a forward pre-hook on GPT-2 attn.c_proj."""

    def __init__(self, model):
        self.model = model
        self.hooks = []
        self.n_heads = model.config.n_head
        self.head_dim = model.config.n_embd // self.n_heads

    def ablation_hook(self, head_indices):
        def hook(module, input):
            modified_input = input[0].clone()
            for h_idx in head_indices:
                start = h_idx * self.head_dim
                end = (h_idx + 1) * self.head_dim
                modified_input[:, :, start:end] = 0
            return (modified_input,)

        return hook

    def apply_ablation(self, layer_head_dict):
        self.remove_hooks()
        for layer_idx, heads in layer_head_dict.items():
            target_module = self.model.transformer.h[layer_idx].attn.c_proj
            h = target_module.register_forward_pre_hook(self.ablation_hook(heads))
            self.hooks.append(h)

    def remove_hooks(self):
        for h in self.hooks:
            h.remove()
        self.hooks = []


In [ ]:
def run_experiment(data_path=DATA_PATH, output_csv=OUTPUT_CSV, model_name=MODEL_NAME):
    tokenizer = GPT2Tokenizer.from_pretrained(model_name)
    model = GPT2LMHeadModel.from_pretrained(model_name, output_attentions=True)
    model.eval()
    ablator = HeadAblator(model)

    df = pd.read_csv(data_path)
    results = []

    for idx, row in df.iterrows():
        prompt = row["Prompt"]
        subject = row["Subject Word(s)"]
        target_token_ids = get_target_token_ids(row["Target Token"], tokenizer)

        # Baseline forward (for attentions)
        inputs = tokenizer(prompt, return_tensors="pt")
        with torch.no_grad():
            outputs = model(**inputs)

        baseline_prob = prob_of_target_sequence(prompt, target_token_ids, model, tokenizer)

        subj_idx = get_subject_token_index(prompt, subject, tokenizer)

        # Mapping: attention from last token to subject token
        all_attentions = torch.stack(outputs.attentions)  # (layers, batch, heads, seq, seq)
        weights = all_attentions[:, 0, :, -1, subj_idx]   # (layers, heads)

        flat_weights = weights.flatten()
        top_indices = torch.argsort(flat_weights, descending=True)

        # Condition A: single strongest head
        best_flat_idx = top_indices[0].item()
        best_layer, best_head = divmod(best_flat_idx, 12)

        # Condition B: top 3 heads
        top3_flat = top_indices[:3].tolist()
        cond_b_heads = {}
        for f_idx in top3_flat:
            l, h = divmod(int(f_idx), 12)
            cond_b_heads.setdefault(l, []).append(h)

        # Condition C: all heads with attention > 0.25
        cond_c_flat = torch.where(flat_weights > 0.25)[0].tolist()
        cond_c_heads = {}
        for f_idx in cond_c_flat:
            l, h = divmod(int(f_idx), 12)
            cond_c_heads.setdefault(l, []).append(h)

        def measure_delta(heads_dict):
            if not heads_dict:
                return 0.0
            ablator.apply_ablation(heads_dict)
            p_int = prob_of_target_sequence(prompt, target_token_ids, model, tokenizer)
            ablator.remove_hooks()
            if baseline_prob == 0.0:
                return 0.0
            return (baseline_prob - p_int) / baseline_prob

        delta_a = measure_delta({best_layer: [best_head]})
        delta_b = measure_delta(cond_b_heads)
        delta_c = measure_delta(cond_c_heads)

        results.append(
            {
                "prompt_id": idx,
                "baseline_prob": baseline_prob,
                "condition_a_delta": delta_a,
                "condition_b_delta": delta_b,
                "condition_c_delta": delta_c,
                "max_head_layer": best_layer,
                "max_head_index": best_head,
            }
        )

    out_df = pd.DataFrame(results)
    out_df.to_csv(output_csv, index=False)
    return out_df


In [ ]:
# Part A — run the full experiment and write ID_results.csv
results_df = run_experiment()
print(f"Saved results to {OUTPUT_CSV}. Rows: {len(results_df)}")
results_df.head()


In [ ]:
# Part B — basic analysis (you can use this for the report narrative)
orig_df = pd.read_csv(DATA_PATH)
res_df = pd.read_csv(OUTPUT_CSV)
res_df["Domain"] = orig_df["Domain"]

print("Top 5 layers containing the max head (Condition A):")
print(res_df["max_head_layer"].value_counts().head(5))

print("\nAverage delta by Domain (Condition A):")
print(res_df.groupby("Domain")["condition_a_delta"].mean().sort_values(ascending=False))


In [ ]:
def plot_required_top5_before_after(prompt, heads_to_ablate, model, tokenizer, title=None):
    """Required visualization: compare baseline top-5 tokens before vs after ablation."""
    inputs = tokenizer(prompt, return_tensors="pt")

    with torch.no_grad():
        out_orig = model(**inputs)
    logits_orig = out_orig.logits[0, -1, :]
    probs_orig = torch.softmax(logits_orig, dim=-1)

    top_ids = torch.topk(probs_orig, 5).indices.tolist()
    tokens = [tokenizer.decode([tid]) for tid in top_ids]
    before = [probs_orig[tid].item() for tid in top_ids]

    ablator = HeadAblator(model)
    ablator.apply_ablation(heads_to_ablate)
    with torch.no_grad():
        out_int = model(**inputs)
    ablator.remove_hooks()

    probs_int = torch.softmax(out_int.logits[0, -1, :], dim=-1)
    after = [probs_int[tid].item() for tid in top_ids]

    x = np.arange(len(tokens))
    width = 0.38

    plt.figure(figsize=(10, 5))
    plt.bar(x - width / 2, before, width, label="Original")
    plt.bar(x + width / 2, after, width, label="Post-Ablation")
    plt.xticks(x, tokens, rotation=20, ha="right")
    plt.ylabel("Probability")
    plt.ylim(0, 1.0)
    plt.title(title or f"Top-5 next-token probabilities (before vs after)\nHeads: {heads_to_ablate}")
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# Required visualization — pick one example (here: max Condition A impact)

# Load model once for plotting
plot_tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
plot_model = GPT2LMHeadModel.from_pretrained(MODEL_NAME, output_attentions=True)
plot_model.eval()

res_df = pd.read_csv(OUTPUT_CSV)
orig_df = pd.read_csv(DATA_PATH)

best_example_idx = res_df["condition_a_delta"].idxmax()
row = res_df.loc[best_example_idx]

prompt_id = int(row["prompt_id"])
layer = int(row["max_head_layer"])
head = int(row["max_head_index"])

prompt = orig_df.iloc[prompt_id]["Prompt"]
domain = orig_df.iloc[prompt_id]["Domain"]

print(f"Selected example prompt_id={prompt_id} domain={domain}")
print("Prompt:", prompt)
print(f"Ablating head {head} in layer {layer}")

plot_required_top5_before_after(
    prompt=prompt,
    heads_to_ablate={layer: [head]},
    model=plot_model,
    tokenizer=plot_tokenizer,
    title=f"Required visualization — domain: {domain} (Condition A)",
)
